# Data processing
This will consist of the data processing one after another. Ie run these in order and you will recreate the database in the /processed_data folder.

Info we only want to run this on a small scale to check that our function are working. We will be using all of the nodes that are adjacent to leaf nodes and doing inference on them as there are much fewer of these.

Objective we want to store our file as a parquet file format. This helps reduce the size since we following columns:
# QuestionId,
# UserId
# AnswerId
# IsCorrect
# Gender
# TimeTaken for the student to complete the question 
# Age
# All of the subject metadata file in the /data/metadata/ folder titled subject_metadata.csv
So we determine the gender form the metadata/student_metadata_task_* .csv 
then you can also determine the age when they did the test by looking for the first answer they completed int answer_metadata_task_*.csv and their date of birt in the student meta data
To get the time taken subtract the differences in times between the answer meta data csv responses for a student and the 
Note for now we want the adjacent to leaves in the question metadata (look at the structure in the subject meta data) that is they form a tree and we want the nodes that are adjacent to the leaves of that tree look.

Be sure to include a percentage bar of how much the programm is complete that regurally updates so i can see if the program paused while runnnign.

First try work each of these colums for 5 answers int the db

In [ ]:
from __future__ import annotations

import ast
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data" / "metadata").exists():
    ROOT = ROOT.parent

SUBJECT_CSV    = ROOT / "data" / "metadata" / "subject_metadata.csv"
QUESTION_CSVS  = [
    ROOT / "data" / "metadata" / "question_metadata_task_1_2.csv",
    ROOT / "data" / "metadata" / "question_metadata_task_3_4.csv",
]
STUDENT_CSVS   = [
    ROOT / "data" / "metadata" / "student_metadata_task_1_2.csv",
    ROOT / "data" / "metadata" / "student_metadata_task_3_4.csv",
]
ANSWER_CSVS    = [
    ROOT / "data" / "metadata" / "answer_metadata_task_1_2.csv",
    ROOT / "data" / "metadata" / "answer_metadata_task_3_4.csv",
]
TRAIN_CSVS     = [
    ROOT / "data" / "train_data" / "train_task_1_2.csv",
    ROOT / "data" / "train_data" / "train_task_3_4.csv",
]
OUTPUT_PARQUET  = ROOT / "processing" / "processed_data" / "full_task_all.parquet"
CHUNK_SIZE      = 50_000


def print_progress(current: int, total: int, *, prefix: str = "Progress", width: int = 40) -> None:
    if total <= 0:
        return
    fraction = min(max(current / total, 0.0), 1.0)
    filled   = int(width * fraction)
    bar      = "#" * filled + "-" * (width - filled)
    print(f"\r{prefix}: [{bar}] {fraction * 100:6.2f}% ({current:,}/{total:,})", end="")
    if current >= total:
        print()


def parse_subject_chain(raw: str) -> list[int]:
    try:
        value = ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return []
    return [int(s) for s in value] if isinstance(value, list) else []


def load_subject_tree(
    subject_csv: Path,
) -> tuple[dict[int, str], dict[int, int], set[int], list[int], dict[int, int]]:
    subject_name_by_id: dict[int, str]      = {}
    parent_by_child:    dict[int, int]      = {}
    children_by_parent: dict[int, set[int]] = defaultdict(set)

    with subject_csv.open("r", encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            sid = int(row["SubjectId"])
            subject_name_by_id[sid] = row["Name"].strip()
            pid_raw = row.get("ParentId", "")
            if pid_raw and pid_raw != "NULL":
                pid = int(pid_raw)
                parent_by_child[sid] = pid
                children_by_parent[pid].add(sid)

    all_ids          = set(subject_name_by_id)
    leaf_subject_ids = all_ids - set(children_by_parent)

    leaf_parent_ids: list[int] = sorted(
        {pid for sid, pid in parent_by_child.items() if sid in leaf_subject_ids}
    )
    leaf_parent_index: dict[int, int] = {pid: i for i, pid in enumerate(leaf_parent_ids)}

    return subject_name_by_id, parent_by_child, leaf_subject_ids, leaf_parent_ids, leaf_parent_index


def load_question_leaf_parent_map(
    question_csvs: list[Path],
    parent_by_child: dict[int, int],
    leaf_subject_ids: set[int],
) -> dict[int, set[int]]:
    question_to_leaf_parents: dict[int, set[int]] = {}

    for csv_path in question_csvs:
        total = sum(1 for _ in csv_path.open("r", encoding="utf-8-sig", newline="")) - 1
        with csv_path.open("r", encoding="utf-8-sig", newline="") as fh:
            for idx, row in enumerate(csv.DictReader(fh), start=1):
                qid   = int(row["QuestionId"])
                chain = set(parse_subject_chain(row["SubjectId"]))
                leaf_parents: set[int] = set()
                for sid in chain:
                    if sid in leaf_subject_ids:
                        pid = parent_by_child.get(sid)
                        if pid is not None:
                            leaf_parents.add(pid)
                question_to_leaf_parents[qid] = leaf_parents
                if idx == 1 or idx % 5000 == 0 or idx == total:
                    print_progress(idx, total, prefix=f"Questions {csv_path.name}")
        print()

    return question_to_leaf_parents


def load_student_lookup(
    student_csvs: list[Path],
) -> tuple[dict[int, int | None], dict[int, pd.Timestamp | None]]:
    gender_by_user: dict[int, int | None]          = {}
    dob_by_user:    dict[int, pd.Timestamp | None] = {}

    for csv_path in student_csvs:
        total = sum(1 for _ in csv_path.open("r", encoding="utf-8-sig", newline="")) - 1
        with csv_path.open("r", encoding="utf-8-sig", newline="") as fh:
            for idx, row in enumerate(csv.DictReader(fh), start=1):
                uid = int(row["UserId"])
                g   = row.get("Gender", "")
                dob = row.get("DateOfBirth", "")
                gender_by_user[uid] = int(g) if g not in {"", None} else None
                dob_by_user[uid]    = pd.to_datetime(dob, errors="coerce") if dob else None
                if idx == 1 or idx % 25000 == 0 or idx == total:
                    print_progress(idx, total, prefix=f"Students {csv_path.name}")
        print()

    return gender_by_user, dob_by_user


def load_answer_times(
    answer_csvs: list[Path],
) -> dict[int, pd.Timestamp]:
    answer_time_by_id: dict[int, pd.Timestamp] = {}

    for csv_path in answer_csvs:
        total = sum(1 for _ in csv_path.open("r", encoding="utf-8-sig", newline="")) - 1
        with csv_path.open("r", encoding="utf-8-sig", newline="") as fh:
            for idx, row in enumerate(csv.DictReader(fh), start=1):
                raw = row.get("AnswerId", "")
                try:
                    aid = int(float(raw)) if "." in raw else int(raw)
                except ValueError:
                    continue
                t = pd.to_datetime(row.get("DateAnswered", ""), errors="coerce")
                if pd.notna(t):
                    answer_time_by_id[aid] = t
                if idx == 1 or idx % 50000 == 0 or idx == total:
                    print_progress(idx, total, prefix=f"Answers {csv_path.name}")
        print()

    return answer_time_by_id


def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8-sig", newline="") as fh:
        return sum(1 for _ in fh) - 1


def iter_train_chunks(
    train_csvs: list[Path],
    chunk_size: int,
):
    """Yield (chunk_df, source_path) across both train files."""
    for csv_path in train_csvs:
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size, encoding="utf-8-sig"):
            yield chunk, csv_path


# ── Load all metadata upfront (fits in RAM) ───────────────────────────────────
print("Loading subject tree...")
(
    subject_name_by_id,
    parent_by_child,
    leaf_subject_ids,
    leaf_parent_ids,
    leaf_parent_index,
) = load_subject_tree(SUBJECT_CSV)
K = len(leaf_parent_ids)
print(f"Leaf-parent skill vocabulary size: {K}\n")

question_leaf_parent_map = load_question_leaf_parent_map(
    QUESTION_CSVS, parent_by_child, leaf_subject_ids
)
print()

print("Loading student metadata...")
student_gender_by_id, student_dob_by_id = load_student_lookup(STUDENT_CSVS)
print()

print("Loading answer times...")
answer_time_by_id = load_answer_times(ANSWER_CSVS)
print()

# ── Stream train data in chunks, write parquet incrementally ──────────────────
OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

total_train_rows = sum(count_csv_rows(p) for p in TRAIN_CSVS)
previous_answer_time_by_user: dict[int, pd.Timestamp] = {}
first_answer_time_by_user:    dict[int, pd.Timestamp] = {}

writer      = None
rows_written = 0

for chunk, source_path in iter_train_chunks(TRAIN_CSVS, CHUNK_SIZE):
    meta_rows:   list[dict] = []
    skill_lists: list[list[int]] = []

    for _, row in chunk.iterrows():
        qid        = int(row["QuestionId"])
        uid        = int(row["UserId"])
        aid        = int(row["AnswerId"])
        is_correct = int(row["IsCorrect"])

        answer_time = answer_time_by_id.get(aid)
        if answer_time is not None and uid not in first_answer_time_by_user:
            first_answer_time_by_user[uid] = answer_time

        gender     = student_gender_by_id.get(uid)
        dob        = student_dob_by_id.get(uid)
        first_time = first_answer_time_by_user.get(uid)
        age_years  = None
        if pd.notna(dob) and first_time is not None:
            age_years = (first_time - dob).days / 365.25

        time_taken_seconds = None
        if answer_time is not None:
            prev = previous_answer_time_by_user.get(uid)
            if prev is not None:
                time_taken_seconds = (answer_time - prev).total_seconds()
            previous_answer_time_by_user[uid] = answer_time

        active_skills = sorted(
            leaf_parent_index[pid] + 1
            for pid in question_leaf_parent_map.get(qid, set())
            if pid in leaf_parent_index
        )

        # gender: 0=unknown/missing, 1 and 2 are the coded values
        gender_stan = int(gender) if gender is not None else 0

        meta_rows.append({
            "QuestionId": qid,
            "UserId":     uid,
            "AnswerId":   aid,
            "IsCorrect":  is_correct,
            "Gender":     np.int8(gender_stan),
            "TimeTaken":  time_taken_seconds,
            "Age":        age_years,
        })
        skill_lists.append(active_skills)

    meta_df = pd.DataFrame(meta_rows)
    table = pa.table({
        **{col: meta_df[col].to_numpy() for col in meta_df.columns},
        "skill_indices": pa.array(skill_lists, type=pa.list_(pa.int32())),
    })

    if writer is None:
        vocab        = {str(i + 1): subject_name_by_id[pid] for i, pid in enumerate(leaf_parent_ids)}
        table        = table.replace_schema_metadata({
            b"skill_vocab_json": str(vocab).encode(),
            b"K":               str(K).encode(),
        })
        writer = pq.ParquetWriter(OUTPUT_PARQUET, table.schema, compression="zstd")

    writer.write_table(table)
    rows_written += len(meta_rows)
    print_progress(rows_written, total_train_rows, prefix="Writing parquet")

if writer:
    writer.close()

print(f"\n\nDone. Rows written: {rows_written:,} | K={K} | Output: {OUTPUT_PARQUET}")

Loading subject tree...
Leaf-parent skill vocabulary size: 69

Questions question_metadata_task_1_2.csv: [########################################] 100.00% (27,613/27,613)

Questions question_metadata_task_3_4.csv: [########################################] 100.00% (948/948)


Loading student metadata...
Students student_metadata_task_1_2.csv: [########################################] 100.00% (118,971/118,971)

Students student_metadata_task_3_4.csv: [########################################] 100.00% (6,148/6,148)


Loading answer times...
Answers answer_metadata_task_1_2.csv: [###########-----------------------------]  28.99% (5,750,000/19,834,820)

In [8]:
from pathlib import Path
import pandas as pd

output_path = Path.cwd()
while output_path != output_path.parent and not (output_path / "processing" / "processed_data" / "sample_task_1_2.parquet").exists():
    output_path = output_path.parent

parquet_file = output_path / "processing" / "processed_data" / "sample_task_1_2.parquet"
if parquet_file.exists():
    df = pd.read_parquet(parquet_file)
    print(f"Loaded {len(df)} rows from {parquet_file}")
    display(df.head())
else:
    print(f"Parquet file not found yet: {parquet_file}")

Loaded 5 rows from /Users/atlasbuchholz/PycharmProjects/UBC/STAT_405_Project/processing/processed_data/sample_task_1_2.parquet


,QuestionId,UserId,AnswerId,IsCorrect,Gender,TimeTaken,Age,skill_indices
0,16997,65967,12453206,0,1,None,NaN,[65]
1,16531,62121,15686710,1,1,None,NaN,[53]
2,15911,50013,13598796,0,1,None,NaN,[9]
3,1701,104909,10511925,0,1,None,14.212183,[44]
4,22896,21748,941747,0,1,None,NaN,[44]


In [6]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_FILE = ROOT / "processing" / "processed_data" / "sample_task_1_2.parquet"
OUTPUT_DIR = ROOT / "processing" / "processed_data" / "abilities_parquet"
CHUNK_SIZE = 50_000
SAMPLE_LIMIT = 5
META_COLS = ["QuestionId", "UserId", "AnswerId", "IsCorrect", "TimeTaken"]
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read sample parquet once to identify ability columns
sample_df = pd.read_parquet(INPUT_FILE)
ability_cols = [c for c in sample_df.columns if c not in META_COLS]

print(f"Found {len(ability_cols)} ability columns")
print(f"Processing in chunks of {CHUNK_SIZE:,}...")

writer = None
total_rows = 0
print(f"Running sample build on {len(sample_df)} rows...")

for chunk_num, chunk in enumerate([sample_df]):
    # Pivot to long format, keeping only active abilities
    long = (
        chunk
        .melt(id_vars=META_COLS, value_vars=ability_cols, var_name="ability", value_name="value")
        .query("value != 0")
        .drop(columns="value")
        .reset_index(drop=True)
    )

    # Cast to efficient types
    long["ability"] = long["ability"].astype("category")
    long["IsCorrect"] = long["IsCorrect"].astype("int8")

    # Write parquet chunk
    table = pa.Table.from_pandas(long)
    if writer is None:
        writer = pq.ParquetWriter(
            OUTPUT_DIR / "abilities.parquet",
            table.schema,
            compression="zstd",
            compression_level=9
        )
    writer.write_table(table)

    total_rows += len(long)
    print_progress(chunk_num + 1, 1, prefix="Building sample parquet")

if writer:
    writer.close()

print(f"\nDone. Total non-zero rows: {total_rows:,}")
print(f"Output: {OUTPUT_DIR / 'abilities.parquet'}")

# ── Check output size ─────────────────────────────────────────────────────────
size_mb = (OUTPUT_DIR / "abilities.parquet").stat().st_size / 1e6
print(f"File size: {size_mb:.1f} MB")

Found 74 ability columns
Processing in chunks of 50,000...
Running sample build on 5 rows...
Building sample parquet: [########################################] 100.00% (1/1)

Done. Total non-zero rows: 15
Output: /Users/atlasbuchholz/PycharmProjects/UBC/STAT_405_Project/processing/processed_data/abilities_parquet/abilities.parquet
File size: 0.0 MB
